# سبق 05 - ایجنٹک RAG


## ترتیب

یہ نوٹ بک مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے ایجنٹک RAG (ریکوریویل-آگمینٹڈ جنریشن) پیٹرن کو ظاہر کرتی ہے۔

**ضروریات:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — آپ کی Azure AI سرچ سروس کا اینڈپوائنٹ
- `AZURE_SEARCH_API_KEY` — آپ کی Azure AI سرچ API کلید
- ماحول کی متغیرات کے ذریعے ترتیب دیا گیا Azure OpenAI ڈپلائمنٹ
- Azure CLI کی توثیق شدہ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ایجنٹک RAG کیا ہے؟

روایتی RAG ایک مقررہ طریقہ کار پر عمل کرتا ہے: دستاویزات بازیافت کریں، پھر جواب تیار کریں۔ **ایجنٹک RAG** آگے بڑھتا ہے اور ایجنٹ کو خودمختاری دیتا ہے کہ وہ فیصلہ کرے **کب** اور **کیسے** معلومات بازیافت کرنی ہے۔

ایجنٹک RAG کے ساتھ، ایجنٹ کر سکتا ہے:
- **فیصلہ** کرے کہ سوال کا جواب دینے سے پہلے بازیافت کی ضرورت ہے یا نہیں
- **انتخاب** کرے کہ کون سا ڈیٹا سورس یا ٹول استفسار کرنا ہے
- **جائزہ** لے کہ بازیافت شدہ نتائج کیسے ہیں اور اگر پہلی کوشش ناکافی ہو تو مزید بازیافت کی جائے
- **معلومات کو ملائے** متعدد بازیافت کے مراحل سے تاکہ ایک مربوط جواب تیار کیا جا سکے

یہ ایجنٹ کو ایک جامد بازیافت-پھر-تیار کرنے کی لائن سے زیادہ لچکدار اور درست بناتا ہے۔


## سرچ ٹول بنانا

ایجنٹک RAG میں، بیرونی ڈیٹا ذرائع کو **ٹولز** کے طور پر لپیٹا جاتا ہے جنہیں ایجنٹ ضرورت کے مطابق بلا سکتا ہے۔ اس سے ایجنٹ کو بازیافت کو صرف ایک اور عمل کے طور پر لینے کی اجازت ملتی ہے جو وہ کر سکتا ہے، بجائے ایک لازمی قدم کے۔

نیچے ہم ایک سفر کے علم کی بنیاد تعریف کرتے ہیں اور اسے ایک ٹول کے طور پر ظاہر کرتے ہیں جسے ایجنٹ منزل کی معلومات تلاش کرنے کے لیے بلا سکتا ہے۔


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## آر اے جی ایجنٹ بنانا

اب ہم ایک ایجنٹ بناتے ہیں جسے ہدایت دی گئی ہے کہ **ہمیشہ جواب دینے سے پہلے معلومات حاصل کرے**۔ ایجنٹ `search_travel_knowledge` ٹول استعمال کرتا ہے تاکہ اپنے جوابات کو اپنے تربیتی ڈیٹا پر انحصار کرنے کے بجائے علم کے ذخیرے میں مستحکم کرے۔


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## تکراری بازیافت — میکر-چیکر پیٹرن

ایجنٹک RAG کا ایک اہم فائدہ **تکراری بازیافت** ہے۔ ایجنٹ کئی مراحل میں تلاش کر سکتا ہے تاکہ اپنی ابتدائی معلومات کی تصدیق کرے، انہیں بہتر بنائے، یا ان میں توسیع کرے — جس طرح "میکر-چیکر" ورک فلو ہوتا ہے:

1. **میکر قدم**: ایجنٹ ابتدائی معلومات حاصل کرتا ہے اور ایک جواب کا مسودہ تیار کرتا ہے۔
2. **چیکر قدم**: ایجنٹ تفصیلات کی تصدیق کرنے یا خالی جگہیں پر کرنے کے لیے اضافی بازیافتیں انجام دیتا ہے۔

نیچے، ایجنٹ سے ایک ایسا سوال پوچھا گیا ہے جس میں ایک سے زیادہ مقامات کا موازنہ کرنا ہوتا ہے، جو اسے کئی بار تلاش کرنے پر مجبور کرتا ہے۔


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ مائیکروسافٹ ایجنٹ فریم ورک استعمال کرتے ہوئے ایک **ایجنٹک RAG** سسٹم کیسے بنایا جائے:

- **ایجنٹک RAG** ایجنٹس کو خود مختار طریقے سے فیصلہ کرنے دیتا ہے کہ معلومات کب حاصل کرنی ہے، جس سے بازیافت متحرک ہوتی ہے، نہ کہ مستحکم۔
- **ٹولز کو ڈیٹا سورس کے طور پر**: بیرونی معلوماتی ذرائع (جیسے Azure AI Search) کو ایسے ٹولز کے طور پر لپیٹا جاتا ہے جنہیں ایجنٹ بلا سکتا ہے۔
- **دوہری بازیافت**: میکر-چیکر پیٹرن ایجنٹ کو متعدد بازیافت کے چکر مکمل کرنے کے قابل بناتا ہے — تلاش کرنا، تصدیق کرنا، اور بہتر بنانا — اس سے پہلے کہ حتمی جواب دیا جائے۔

پیداواری ماحول میں، آپ ان میموری `TRAVEL_KNOWLEDGE_BASE` کو حقیقی Azure AI Search انڈیکس سے بدلیں گے تاکہ بڑے پیمانے پر سفر کے دستاویزات کی بازیافت کی جا سکے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
